# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'deepseek-chat'
openai = OpenAI(base_url="https://api.deepseek.com")

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [4]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [5]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in a valid JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in a valid JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [15]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [16]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [17]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/moonshotai/Kimi-K2-Instruct',
 '/mistralai/Voxtral-Mini-3B-2507',
 '/mistralai/Voxtral-Small-24B-2507',
 '/HuggingFaceTB/SmolLM3-3B',
 '/LGAI-EXAONE/EXAONE-4.0-32B',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/black-forest-labs/FLUX.1-Kontext-Dev',
 '/spaces/llamameta/Grok-4-heavy-free',
 '/spaces/multimodalart/wan2-1-fast',
 '/spaces/FunAudioLLM/ThinkSound',
 '/spaces',
 '/datasets/NousResearch/Hermes-3-Dataset',
 '/datasets/common-pile/caselaw_access_project',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/microsoft/rStar-Coder',
 '/datasets/HuggingFaceTB/smoltalk2',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/gramma

In [18]:
get_links("https://huggingface.co")

{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'join page', 'url': 'https://huggingface.co/join'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://w

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [19]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [20]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'join page', 'url': 'https://huggingface.co/join'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/co

In [28]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'models', 'url': 'https://huggingface.co/models'}, {'type': 'datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'join', 'url': 'https://huggingface.co/join'}, {'type': 'chat', 'url': 'https://huggingface.co/chat'}, {'type': 'brand', 'url': 'https://huggingface.co/brand'}, {'type': 'learn', 'url': 'https://huggingface.co/learn'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'discord', 'url': 'https://huggingface.co/join/discord'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2-Instruct\nUpdated\n1 day ago\n•\n126k\n•\n1.5k\nmistralai/Voxtral-Mini-3B-2507\nUpdated\n1 day ago\n•\n11.2k\n•\n329\nmistralai/Voxtral-Small-24B-2507\nUpdated\n1 day ago\n•\n281\n•\n283\nHuggingFaceTB/SmolLM3-3B\nUpdated\n1 day ago\n•\n212k\n•\n535\nLGAI-EXAONE/EXAONE-4.0-32B\nUpdated\n1 day ago\n•\n206k\n•\n167\nBrowse 1M+ models\nSpaces\nRunning\n10.4k\n10.4k\nDeepSit

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'models', 'url': 'https://huggingface.co/models'}, {'type': 'datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'join', 'url': 'https://huggingface.co/join'}, {'type': 'docs', 'url': 'https://huggingface.co/docs'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'brand', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'discord', 'url': 'https://huggingface.co/join/discord'}]}


```markdown
# 🌟 Hugging Face: The AI Community Building the Future

## 🚀 **Who We Are**
Hugging Face is the leading open collaboration platform for the machine learning community, empowering developers, researchers, and enterprises to build, share, and deploy cutting-edge AI models, datasets, and applications.

### **Our Mission**
To democratize AI by fostering collaboration and innovation through open-source tools and community-driven resources.

---

## 🔥 **What We Offer**
### **1. Models**
- **1M+ models** spanning text, image, video, audio, and 3D.
- **Trending models** updated daily (e.g., `moonshotai/Kimi-K2-Instruct`, `mistralai/Voxtral`).
- **State-of-the-art frameworks** like Transformers, Diffusers, and PEFT.

### **2. Datasets**
- **250k+ datasets** for training and benchmarking.
- **Popular datasets** like `Hermes-3-Dataset`, `awesome-chatgpt-prompts`.

### **3. Spaces (AI Apps)**
- **400k+ applications** hosted, from text generation to video synthesis.
- **Examples**: DeepSeek, FLUX Kontext, Grok 4 Heavy.

### **4. Open-Source Tools**
- **Transformers**: 147K+ stars on GitHub.
- **Diffusers**: For diffusion models.
- **Tokenizers, Datasets, TRL, and more**.

---

## 🌍 **Community & Culture**
- **Collaborative**: A global hub for ML practitioners to share and learn.
- **Inclusive**: Open to researchers, startups, and enterprises.
- **Innovative**: Home to bleeding-edge AI projects.

### **Who Uses Hugging Face?**
- **50,000+ organizations**, including:
  - **Google, Microsoft, Meta, Intel, Amazon**
  - **Grammarly, Writer, AI2 (Allen Institute for AI)**

---

## 💼 **Enterprise Solutions**
For teams scaling AI with security and efficiency:
- **Compute**: GPU-powered Inference Endpoints ($0.60/hour).
- **Team & Enterprise Plans**: From **$20/user/month**.
  - SSO, audit logs, private datasets, priority support.

---

## 🛠 **Careers at Hugging Face**
Join a mission-driven team shaping the future of AI:
- **Open roles**: Engineering, Research, Product, and more.
- **Culture**: Fast-paced, open-source ethos, remote-friendly.
- **Perks**: Work with cutting-edge tech alongside top ML talent.

🔗 **Explore jobs**: [Hugging Face Careers](https://huggingface.co/jobs)

---

## ✨ **Why Choose Hugging Face?**
- **Largest ML repository** (models, datasets, apps).
- **Open-source leadership** (Transformers, Diffusers, etc.).
- **Trusted by industry giants and indie developers alike**.

📢 **Join the future of AI—**[Sign Up Free](https://huggingface.co) | [Enterprise Inquiry](https://huggingface.co/enterprise)

``` 

This brochure highlights Hugging Face’s offerings, culture, and impact while catering to users, enterprises, and potential recruits. Let me know if you'd like adjustments! 🚀

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'join page', 'url': 'https://huggingface.co/join'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/co


# 🌟 Hugging Face: The AI Community Building the Future  

## 🚀 **Who We Are**  
Hugging Face is the leading open collaboration platform for the machine learning community, empowering developers, researchers, and enterprises to **build, share, and deploy AI models, datasets, and applications**.  

### **Our Mission**  
To democratize AI by providing tools, resources, and a collaborative ecosystem where innovation thrives.  

---

## 🔍 **What We Offer**  

### **1. Open Collaboration Hub**  
- **1M+ models** (text, image, video, audio, 3D)  
- **250k+ datasets** for diverse ML tasks  
- **400k+ AI applications** (Spaces) for experimentation  

### **2. Cutting-Edge Tools**  
- **Transformers**: State-of-the-art ML frameworks  
- **Diffusers**: Advanced diffusion models  
- **Datasets & Tokenizers**: Optimized for research/production  
- **Enterprise Solutions**: Secure, scalable AI infrastructure  

### **3. Community-Driven Innovation**  
- Host and collaborate on **public/private projects**  
- Build your **ML portfolio** and gain visibility  
- Join **50,000+ organizations** (Google, Meta, Microsoft, Intel, and more)  

---

## 💼 **For Enterprises & Teams**  
- **Inference Endpoints**: GPU-optimized deployment  
- **Enterprise Plans**:  
  - SSO, audit logs, priority support  
  - Starting at **$20/user/month**  
- **Private datasets & models** with granular access controls  

---

## 🌱 **Join the Movement**  

### **For Developers & Researchers**  
- Contribute to **open-source projects** (Transformers, TRL, PEFT)  
- Showcase work in **Spaces** or **HuggingChat**  
- Access **free resources** and tutorials  

### **For Job Seekers**  
- Work at the forefront of AI with a **remote-first, inclusive culture**  
- Roles in **engineering, research, and community growth**  
- Check our [Jobs Page](https://huggingface.co/jobs)  

---

## 🤝 **Trusted By**  
![Logos: Google, Meta, Microsoft, Intel, Grammarly, Writer, AI2]  

🔗 **Explore More**: [huggingface.co](https://huggingface.co) | [GitHub](https://github.com/huggingface) | [Twitter](https://twitter.com/huggingface)  

*"The Home of Machine Learning"*  
 

This brochure highlights Hugging Face’s **open ethos**, **technical leadership**, and **community impact** while catering to developers, enterprises, and recruits. Let me know if you'd like adjustments!

In [29]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'models', 'url': 'https://huggingface.co/models'}, {'type': 'datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'join', 'url': 'https://huggingface.co/join'}, {'type': 'docs', 'url': 'https://huggingface.co/docs'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'brand', 'url': 'https://huggingface.co/brand'}, {'type': 'careers', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'github', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'discord', 'url': 'https://huggingface.co/join/discord'}]}


# **🤗 Welcome to Hugging Face – Where AI Gets Hugs!**  

## **🚀 The AI Playground for Geeks, Nerds, & Future Overlords**  

At **Hugging Face**, we don’t just build AI—we *hug* it. We’re the ultimate **ML community** where models, datasets, and AI applications come to party. Think of us as the **GitHub of AI**, but with more emojis and fewer existential crises about code.  

### **🔥 What’s Cooking?**  
- **1M+ Models** – Because why settle for one when you can have a *million*?  
- **400K+ AI Apps** – From chatbots to deepfake generators (use responsibly, please).  
- **250K+ Datasets** – For when you need *all* the data but don’t want to Google it.  

### **💼 Who’s Using Us? (Spoiler: Everyone)**  
Big names like **Google, Meta, Microsoft, Intel**, and even **Grammarly** (because AI needs good grammar too).  

### **💻 Open Source? More Like Open *Awesome***  
We build **foundational ML tools** with the community, including:  
- **Transformers** – Because *"Attention is All You Need"* (and coffee).  
- **Diffusers** – For when you want AI to paint like Picasso.  
- **Tokenizers** – Faster than your ex moving on.  

### **💰 Pricing? We Got You**  
- **Free Tier** – For broke geniuses.  
- **$0.60/hour for GPU** – Cheaper than therapy.  
- **Enterprise Plans** – For when your AI needs a *butler*.  

### **🤝 Join the Cult—Uh, Community!**  
Want to **build, share, or just lurk**? Sign up and start hugging (AI models, not strangers).  

👉 **[Sign Up Now](https://huggingface.co)** – Before the robots take over.  

**P.S.** We’re hiring! If you love AI more than sleep, check out our **[Jobs](https://huggingface.co/jobs)** page. 🚀  

---  
*"Hugging Face: Because the future of AI shouldn’t be lonely."* 🤖💙

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>